In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def conv_weight_grid_image(weight_oiHW: torch.Tensor, max_out: int = 32) -> wandb.Image:
    w = weight_oiHW.detach().float().cpu()
    out_ch, in_ch, k_h, k_w = w.shape
    n = min(int(max_out), int(out_ch))
    w = w[:n]

    if in_ch >= 3:
        w = w[:, :3]
    else:
        w = w[:, :1].repeat(1, 3, 1, 1)

    x = w.permute(0, 2, 3, 1).contiguous().numpy()
    x = x - x.min()
    denom = float(x.max() - x.min())
    if denom > 1e-8:
        x = x / denom

    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    grid = np.zeros((rows * k_h, cols * k_w, 3), dtype=np.float32)
    for idx in range(n):
        r = idx // cols
        c = idx % cols
        grid[r * k_h : (r + 1) * k_h, c * k_w : (c + 1) * k_w] = x[idx]

    fig, ax = plt.subplots(figsize=(min(cols * 0.35, 12), min(rows * 0.35, 12)))
    ax.imshow(grid.clip(0, 1), interpolation="nearest")
    ax.axis("off")
    fig.tight_layout(pad=0)
    out = wandb.Image(fig)
    plt.close(fig)
    return out


In [ ]:
FeatureMaps = list[tuple[str, torch.Tensor]]


class CachesDetachedConvWeights(nn.Module):
    """Detached copies of each Conv2d weight (same device/dtype as the live weight).

    Call `rebuild_conv_weight_snapshots()` after submodules exist. In `forward`, call
    `refresh_conv_weight_snapshots_if_eval()` so copies refresh only under eval / inference.
    Dict tensors follow `model.to(...)` via `_apply`.
    """

    def __init__(self) -> None:
        super().__init__()
        self.conv_weight_snapshots: dict[str, torch.Tensor] = {}

    def rebuild_conv_weight_snapshots(self) -> None:
        self.conv_weight_snapshots = {}
        for module_name, module in self.named_modules():
            if isinstance(module, nn.Conv2d):
                self.conv_weight_snapshots[f"{module_name}.weight"] = torch.zeros_like(module.weight.detach())

    def _apply(self, fn):
        r = super()._apply(fn)
        for key, tensor in list(self.conv_weight_snapshots.items()):
            self.conv_weight_snapshots[key] = fn(tensor)
        return r

    @torch.no_grad()
    def refresh_conv_weight_snapshots_if_eval(self) -> None:
        if torch.is_grad_enabled() and not torch.is_inference_mode_enabled():
            return
        for module_name, module in self.named_modules():
            if isinstance(module, nn.Conv2d):
                key = f"{module_name}.weight"
                self.conv_weight_snapshots[key].copy_(module.weight.detach())


class CNN(CachesDetachedConvWeights):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )
        self.rebuild_conv_weight_snapshots()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, FeatureMaps]:
        fms: FeatureMaps = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if isinstance(layer, nn.Conv2d):
                fms.append((f"features.conv{idx}", x))
        logits = self.classifier(x)
        self.refresh_conv_weight_snapshots_if_eval()
        return logits, fms


class VGG16(CachesDetachedConvWeights):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()

        def conv_block(in_ch: int, out_ch: int, n_conv: int) -> list[nn.Module]:
            layers: list[nn.Module] = []
            ch = in_ch
            for _ in range(n_conv):
                layers.append(nn.Conv2d(ch, out_ch, kernel_size=3, padding=1))
                layers.append(nn.ReLU(inplace=True))
                ch = out_ch
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            return layers

        self.features = nn.Sequential(
            *conv_block(3, 64, 2),
            *conv_block(64, 128, 2),
            *conv_block(128, 256, 3),
            *conv_block(256, 512, 3),
            *conv_block(512, 512, 3),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes),
        )
        self.rebuild_conv_weight_snapshots()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, FeatureMaps]:
        fms: FeatureMaps = []
        conv_idx = 0
        for layer in self.features:
            x = layer(x)
            if isinstance(layer, nn.Conv2d):
                fms.append((f"features.conv{conv_idx}", x))
                conv_idx += 1
        x = x.flatten(1)
        logits = self.classifier(x)
        self.refresh_conv_weight_snapshots_if_eval()
        return logits, fms


class BasicBlock(nn.Module):
    expansion: int = 1

    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        stride: int,
        downsample: nn.Module | None,
    ) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.downsample = downsample

    def forward(self, x: torch.Tensor, fms: FeatureMaps, name_prefix: str) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        fms.append((f"{name_prefix}.conv1", out))
        out = self.bn1(out)
        out = F.relu(out, inplace=True)

        out = self.conv2(out)
        fms.append((f"{name_prefix}.conv2", out))
        out = self.bn2(out)

        if self.downsample is not None:
            conv_ds, bn_ds = self.downsample[0], self.downsample[1]
            identity = conv_ds(x)
            fms.append((f"{name_prefix}.downsample.conv", identity))
            identity = bn_ds(identity)

        out = out + identity
        out = F.relu(out, inplace=True)
        return out


class ResNet18(CachesDetachedConvWeights):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, num_blocks=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        self.rebuild_conv_weight_snapshots()

    def _make_layer(self, in_ch: int, out_ch: int, num_blocks: int, stride: int) -> nn.ModuleList:
        blocks = nn.ModuleList()
        for block_idx in range(num_blocks):
            s = stride if block_idx == 0 else 1
            downsample = None
            if s != 1 or in_ch != out_ch:
                downsample = nn.Sequential(
                    nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=s, bias=False),
                    nn.BatchNorm2d(out_ch),
                )
            blocks.append(BasicBlock(in_ch, out_ch, stride=s, downsample=downsample))
            in_ch = out_ch
        return blocks

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, FeatureMaps]:
        fms: FeatureMaps = []

        x = self.conv1(x)
        fms.append(("conv1", x))
        x = self.bn1(x)
        x = F.relu(x, inplace=True)

        for i, block in enumerate(self.layer1):
            x = block(x, fms, name_prefix=f"layer1.block{i}")
        for i, block in enumerate(self.layer2):
            x = block(x, fms, name_prefix=f"layer2.block{i}")
        for i, block in enumerate(self.layer3):
            x = block(x, fms, name_prefix=f"layer3.block{i}")
        for i, block in enumerate(self.layer4):
            x = block(x, fms, name_prefix=f"layer4.block{i}")

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        self.refresh_conv_weight_snapshots_if_eval()
        return logits, fms


In [ ]:
print(f"device: {DEVICE}")

ARCHS: dict[str, type[nn.Module]] = {
    "cnn": CNN,
    "vgg16": VGG16,
    "resnet18": ResNet18,
}

arch_name = os.environ.get("CNN_REDUNDANCY_ARCH", "cnn").lower().strip()
ModelCls = ARCHS.get(arch_name, CNN)
model = ModelCls(num_classes=10).to(DEVICE)

run = wandb.init(
    project=os.environ.get("WANDB_PROJECT", "cnn-redundancy"),
    name=os.environ.get("WANDB_RUN_NAME", f"filters-{arch_name}"),
    config={"arch": arch_name, "num_classes": 10, "device": str(DEVICE)},
)

x = torch.randn(4, 3, 32, 32, device=DEVICE)
with torch.inference_mode():
    logits, feature_maps = model(x)

print("logits:", tuple(logits.shape))
print("feature_maps:", len(feature_maps))
print("snapshots:", len(model.conv_weight_snapshots))

for weight_name, weight_snapshot in model.conv_weight_snapshots.items():
    wandb.log({f"filters/{weight_name}": conv_weight_grid_image(weight_snapshot, max_out=32)})

run.finish()
